# Lab 5 — Routing Agent
**Pattern: ROUTING** — from Anthropic's five agentic patterns

---

## What is the Routing pattern?

Before an agent does any work, it first asks: **what kind of task is this?**  
Then it sends the task to a handler that is specifically built for that type.

The router does **not** answer the question. It only decides **who should**.

---

## Real-world analogy

Think about how a hospital works. A patient walks in. The receptionist (the **router**) asks what the problem is and sends them to the right department — A&E, outpatients, or pharmacy. The department (the **specialist handler**) takes over from there. The receptionist doesn't treat the patient.

Same idea here:
- A **technical question** goes to a handler with an engineer's system prompt
- A **billing question** goes to a handler with a support agent's system prompt
- A **general question** goes to a default handler

The same question sent to the wrong handler gets a worse answer.  
That is the entire point of routing.

---

## What you will learn
1. How to classify a query before acting on it
2. How specialist handlers with different system prompts outperform a single generic one
3. How to build a clean router → dispatcher pipeline
4. When routing is worth adding vs. when a single agent is enough

---

## Prerequisites
- `ANTHROPIC_API_KEY` set in your `.env` file
- `anthropic` and `python-dotenv` installed (`pip install anthropic python-dotenv`)

## Step 1 — Setup
Import libraries and initialise the Anthropic client.

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

print("✓ Client ready")

## Step 2 — Specialist Handlers

Each handler is a separate function with its own system prompt tuned for one domain.

> **Key insight:** A technical engineer's system prompt and a billing support agent's system prompt are fundamentally different in tone, depth, and what they prioritise.  
> A single generic system prompt compromises both.

In [ ]:
def handle_technical(query: str) -> str:
    """
    Handles: architecture, code, APIs, infrastructure, debugging.
    System prompt: senior engineer — precise, specific, direct.
    """
    print("  [ROUTER] → Technical Handler")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="""You are a senior cloud architect and engineer.
Answer technical questions with precision. Use specific service names,
version numbers, and command examples where relevant. If there are
tradeoffs, state them clearly. Be direct — engineers don't need padding.""",
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text


def handle_billing(query: str) -> str:
    """
    Handles: pricing, cost, invoices, subscriptions, refunds.
    System prompt: support specialist — empathetic, clear, actionable.
    """
    print("  [ROUTER] → Billing Handler")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system="""You are a billing support specialist.
Answer questions about pricing, costs, and subscriptions clearly.
Always acknowledge the customer's concern first. If you cannot give
an exact figure, explain why and what they should do next.
Keep responses concise and actionable.""",
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text


def handle_general(query: str) -> str:
    """
    Handles: anything that doesn't fit the specialist categories.
    System prompt: generic helpful assistant.
    """
    print("  [ROUTER] → General Handler")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system="You are a helpful assistant. Answer clearly and concisely.",
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text

print("✓ Handlers defined")

## Step 3 — The Router

The router is **itself an LLM call**. It reads the query and returns a classification.

Notice two things:
1. It uses a **very small `max_tokens`** — it only needs one word back
2. The system prompt lists **exactly three categories** with clear definitions

The router is intentionally simple and cheap. The specialists do the expensive work.

In [ ]:
def classify_query(query: str) -> str:
    """
    Returns exactly one of: 'technical', 'billing', 'general'
    
    This is a single, cheap LLM call — just classification, no answering.
    We use max_tokens=64 because we only need one word back.
    """
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=64,
        system="""You are a query classifier. Classify the user's query into exactly
one category. Reply with ONLY the category name — nothing else.

Categories:
  technical  — architecture, code, APIs, debugging, infrastructure, deployment
  billing    — pricing, cost, invoices, subscriptions, payments, refunds
  general    — anything else""",
        messages=[{"role": "user", "content": query}]
    )
    raw = response.content[0].text.strip().lower()
    # Guard: if model returns something unexpected, default to 'general'
    return raw if raw in ("technical", "billing", "general") else "general"

print("✓ Router defined")

## Step 4 — Full Pipeline

`route_and_respond()` ties it together:
1. Classify the query
2. Dispatch to the right handler
3. Return the handler's answer

The caller never knows which handler ran. They just get a better answer.

In [ ]:
def route_and_respond(query: str) -> str:
    """
    Full routing pipeline:
      classify → dispatch → return specialist answer
    """
    print(f"\n── Query: {query[:80]}...")
    
    category = classify_query(query)
    print(f"  [ROUTER] Classified as: '{category}'")
    
    if category == "technical":
        return handle_technical(query)
    elif category == "billing":
        return handle_billing(query)
    else:
        return handle_general(query)

print("✓ Pipeline ready")

## Step 5 — Run the Demos

Four queries, each designed to exercise a different path through the router.

Watch the `[ROUTER] →` line to see which handler gets called.

In [ ]:
# ── Query 1: Technical ──────────────────────────────────────────────────────
query1 = (
    "How do I configure Karpenter to prefer Spot instances "
    "but fall back to On-Demand if Spot capacity is unavailable?"
)
answer1 = route_and_respond(query1)
print(f"\n── Answer:\n{answer1}")

In [ ]:
# ── Query 2: Billing ────────────────────────────────────────────────────────
query2 = (
    "I was charged twice this month for my API subscription. "
    "Can you explain why and how I get a refund?"
)
answer2 = route_and_respond(query2)
print(f"\n── Answer:\n{answer2}")

In [ ]:
# ── Query 3: General ────────────────────────────────────────────────────────
query3 = "What is the difference between a transformer and an RNN in simple terms?"
answer3 = route_and_respond(query3)
print(f"\n── Answer:\n{answer3}")

In [ ]:
# ── Query 4: Edge case — could go either way ────────────────────────────────
# Watch where the router sends this one.
query4 = "Is using GPT-4 more expensive than Claude for high-volume workloads?"
answer4 = route_and_respond(query4)
print(f"\n── Answer:\n{answer4}")

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| The router is ONE cheap LLM call | The classification cost is tiny. The specialist does the expensive work. |
| Different system prompts = different quality | A technical handler and a billing handler are not interchangeable. |
| You can extend routing to cost-optimise | Route simple queries to a smaller model, complex ones to a larger model. |
| The pattern is universal | Every production support system, API gateway, and mixed-use AI product uses this. |

---

**Next lab:** Lab 6 — FSM-Controlled Agent  
You'll make the agent's internal state machine explicit, so you can see exactly where it is in its lifecycle at every moment.